In [1]:
calls = 0
def fib(n):
    global calls
    calls += 1
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)

print(fib(30), "computed in", calls, "calls")

832040 computed in 2692537 calls


In [2]:
from functools import lru_cache


@lru_cache(maxsize=None)
def fib(n):
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)

print(fib(30))
print(fib.cache_info())

832040
CacheInfo(hits=28, misses=31, maxsize=None, currsize=31)


In [3]:
import time

# Three functions with deliberately different complexity classes.
def constant_op(data):          # O(1): touch one element
    return data[0]

def linear_op(data):            # O(n): sum every element
    total = 0
    for x in data:
        total += x
    return total

def quadratic_op(data):         # O(n^2): compare every pair
    count = 0
    for a in data:
        for b in data:
            if a == b:
                count += 1
    return count

def time_it(func, n):
    data = list(range(n))
    start = time.perf_counter()
    func(data)
    return time.perf_counter() - start

sizes = [200, 400, 800, 1600, 3200]
for label, fn in [("O(1)", constant_op), ("O(n)", linear_op), ("O(n^2)", quadratic_op)]:
    row = [f"{time_it(fn, n)*1000:8.3f}" for n in sizes]
    print(f"{label:7}", "  ".join(row), "ms")
print("\nColumns = n:", sizes)

O(1)       0.001     0.003     0.000     0.000     0.000 ms
O(n)       0.008     0.018     0.037     0.074     0.109 ms
O(n^2)     0.706     2.880    10.045    37.080   152.399 ms

Columns = n: [200, 400, 800, 1600, 3200]


## Arrays

* Homogeneous data type
* Contiguous
* Fixed size

**Array costs:**

| Operation | Cost | Why |
|---|---|---|
| index `lst[i]` | O(1) | address arithmetic |
| append | O(1)* | amortised; occasional resize |
| insert at front | **O(n)** | shift everything right |
| delete at front | **O(n)** | shift everything left |
| search by value | O(n) | scan |

In [4]:
sample = [10, 20, 30]
for i, value in enumerate(sample):
    print(f"index {i}: value={value}  id(value)={id(value)}")

index 0: value=10  id(value)=140727643763784
index 1: value=20  id(value)=140727643764104
index 2: value=30  id(value)=140727643764424


## Linked Lists

In [5]:
class Node:
    def __init__(self, value):
        self.value = value
        self.next = None

class LinkedList:
    def __init__(self):
        self.head = None          # entry point to the chain

    def prepend(self, value):     # add at FRONT — O(1), just rewire head
        node = Node(value)
        node.next = self.head
        self.head = node

    def append(self, value):      # add at END — O(n), must walk to the tail
        node = Node(value)
        if self.head is None:
            self.head = node
            return
        cur = self.head
        while cur.next is not None:
            cur = cur.next
        cur.next = node

    def display(self):            # walk the chain, collect values
        out, cur = [], self.head
        while cur is not None:
            out.append(cur.value)
            cur = cur.next
        return out
    
#-----------------------------------------------------------------

chain = LinkedList()
for amount in [1500, 320, 9800]:
    chain.append(amount)
chain.prepend(75)                 # newest-first arrival at the front

print("chain:", chain.display())  # [75, 1500, 320, 9800]

chain: [75, 1500, 320, 9800]


In [6]:
N = 60000

# Array (list): insert at front N times -> each insert shifts everything -> O(n) each
arr = []
start = time.perf_counter()
for i in range(N):
    arr.insert(0, i)              # the expensive line: O(n) shift
arr_time = time.perf_counter() - start

# Linked list: prepend N times -> each is O(1) pointer rewiring
ll = LinkedList()
start = time.perf_counter()
for i in range(N):
    ll.prepend(i)                 # the cheap line: O(1)
ll_time = time.perf_counter() - start

print(f"list.insert(0, x)  x{N}: {arr_time*1000:8.1f} ms   (O(n) each)")
print(f"linkedlist.prepend x{N}: {ll_time*1000:8.1f} ms   (O(1) each)")
print(f"\nlinked list was ~{arr_time/ll_time:.0f}x faster for front insertion")

list.insert(0, x)  x60000:   1630.1 ms   (O(n) each)
linkedlist.prepend x60000:    344.0 ms   (O(1) each)

linked list was ~5x faster for front insertion


| | Array (`list`) | Linked List |
|---|---|---|
| Index access `[i]` | **O(1)** | O(n) — must walk |
| Insert/delete at front | O(n) | **O(1)** |
| Insert/delete at end | O(1)* | O(n) without a tail pointer |
| Memory | compact, contiguous | extra pointer per node |
| Cache behaviour | excellent (locality) | poor (scattered) |


## Stacks (LIFO) & Queues (FIFO)

In [8]:
class Stack:
    """A simple LIFO stack backed by a Python list (top = end of list)."""

    def __init__(self):
        self._items = []          # the end of the list is the "top"

    def push(self, item):
        self._items.append(item)  # O(1) amortized

    def pop(self):
        if self.is_empty():
            raise IndexError("pop from an empty stack")
        return self._items.pop()  # removes and returns the top

    def peek(self):
        if self.is_empty():
            raise IndexError("peek into an empty stack")
        return self._items[-1]    # look at the top without removing

    def is_empty(self):
        return len(self._items) == 0

    def size(self):
        return len(self._items)

    def data(self):
        return len(self._items)

    def __repr__(self):
        # bottom -> top, so the rightmost item is the top
        return f"Stack(bottom -> top: {self._items})"
    
#-------------------------------------------------------------------------------------

s = Stack()
print("Empty at start? ", s.is_empty())   # True

for x in (10, 20, 30):
    s.push(x)
    print(f"pushed {x:>2} -> {s}")

print("Top (peek):     ", s.peek())        # 30, stack unchanged
print("Size:           ", s.size())        # 3

print("popped:         ", s.pop())         # 30
print("popped:         ", s.pop())         # 20
print("After two pops: ", s.data())               # bottom -> top: [10]

print("Empty now?      ", s.is_empty())    # False

Empty at start?  True
pushed 10 -> Stack(bottom -> top: [10])
pushed 20 -> Stack(bottom -> top: [10, 20])
pushed 30 -> Stack(bottom -> top: [10, 20, 30])
Top (peek):      30
Size:            3
popped:          30
popped:          20
After two pops:  1
Empty now?       False


In [9]:
# A rollback log: the last action applied is the first to be undone.
rollback = []
for action in ["debit-500", "credit-200", "fee-15"]:
    rollback.append(action)       # push
    print("pushed:", action)

print("\nundo order (LIFO):")
while rollback:
    print("  undo ->", rollback.pop())   # pop from the top

pushed: debit-500
pushed: credit-200
pushed: fee-15

undo order (LIFO):
  undo -> fee-15
  undo -> credit-200
  undo -> debit-500
